In [293]:
import xarray as xr
import cfgrib
import airportsdata
import pandas as pd
#import numpy as np
#import os, sys
from datetime import datetime, timezone, timedelta
import zoneinfo

# Provides latitude, longitude information for airports based on IATA code
iata_airports = airportsdata.load('IATA')

In [181]:
# Loads the paths used for the weather data, flight data, and the folder where combined data will be delivered

path_grib = '/Users/joaopedrocarvalho/Documents/erdos_spring_2026/weather-data/'
path_flights = '/Users/joaopedrocarvalho/Documents/erdos_spring_2026/airline-data/'
path_dump = '/Users/joaopedrocarvalho/Documents/erdos_spring_2026/combined-data-fix/'

In [190]:
# Takes in IATA airport, returns longitude and latitude

def lon_lat(airport):
    lon = iata_airports[airport]['lon']
    lat = iata_airports[airport]['lat']
    #lon = round(lon_raw * 4)/4
    #lat = round(lat_raw * 4)/4
    return [lon, lat]

In [305]:
# These functions provide UTC timestamps for out flights in the format GRIB requires

def convert_date_step(year, month, day, hhmm, airport):
    # This first step is to prevent a strptime error, since we round the hours after anyways it does not affect outcome
    if hhmm == 2400:
        hhmm = 2359    
    dt = datetime.strptime(str(hhmm).zfill(4), '%H%M')
    dt = dt.replace(year = year, month = month, day = day, tzinfo = zoneinfo.ZoneInfo(iata_airports[airport]['tz']))
    if dt.minute >= 30:
        # Add one hour, then truncate minutes/seconds
        dt = (dt + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)
    else:
        # Just truncate minutes/seconds
        dt = dt.replace(minute=0, second=0, microsecond=0)
    dt = dt.astimezone(timezone.utc)
    if 6 < dt.hour and dt.hour <= 18:
        baseline = dt.replace(hour = 6)
        step = timedelta(hours = dt.hour - 6)
    elif dt.hour > 18:
        baseline = dt.replace(hour = 18)
        step = timedelta(hours = dt.hour - 18)
    elif dt.hour <= 6:
        baseline = (dt - timedelta(days = 1)).replace(hour = 18)
        step = timedelta(hours = dt.hour + 6)
    #print(dt, baseline, step, str(dt.date())+'T'+str(dt.time()), str(baseline.date())+'T'+str(baseline.time()), 'T'+str(step))
    dt = dt.replace(tzinfo = None)
    baseline = baseline.replace(tzinfo = None)
    return [dt, baseline, step]

def convert_arr_date_step(year, month, day, dep_hhmm, arr_hhmm, airport):
    # To account for when a flight arrives the following day
    if dep_hhmm < arr_hhmm:
        return convert_date_step(year, month, day, arr_hhmm, airport)
    else:
        dt = datetime(year = year, month = month, day = day)
        dt = dt + timedelta(days = 1)
        return convert_date_step(dt.year, dt.month, dt.day, arr_hhmm, airport)


In [ ]:
# Takes in the pre-cleaned flight data, does some extra processing, and returns a new CSV file with added columns for departure and arrival weather information

for year in range(2020, 2026):

    # Conditions are based on the months of each year we are using, each month is then treated in the same way
    if year == 2020:

        # Opens flight and weather files, adds departure and arrival longitute and latitude data to flights
        # The GRIB files have the information in two split datasets
        # Excludes any flights whose origin or destination is not in the continental USA
        flights = pd.read_csv(path_flights + 'cleaned_2020_12.csv')
        weather = cfgrib.open_datasets(path_grib + '2020_dec_weather.grib')
        weather_step = weather[0]
        weather_step = weather[1]
        flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
        flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
        in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
        flights = flights[in_window_dep]
        in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
        flights = flights[in_window_arr]

        # Creates UTC time datetimes for indexing into GRIB
        flights[['UTC_CRSDepTime', 'UTC_CRSDepBaseline' , 'UTC_CRSDepStep']] = flights.apply(lambda row: convert_date_step(row['Year'], row['Month'], row['DayofMonth'], row['CRSDepTime'], row['Origin']), 
                                                                                     axis=1, result_type = 'expand')
        flights[['UTC_CRSArrTime', 'UTC_CRSArrBaseline' , 'UTC_CRSArrStep']] = flights.apply(lambda row: convert_arr_date_step(row['Year'], row['Month'], row['DayofMonth'], row['CRSDepTime'], row['CRSArrTime'], row['Origin']), 
                                                                                     axis=1, result_type = 'expand')

        # Creates xarray versions of our dataset, compiles the weather data with same index into new xrrays, transform then back into pandas and merges
        indexer_dep = flights[['dep_longitude', 'dep_latitude','UTC_CRSDepTime', 'UTC_CRSDepBaseline' , 'UTC_CRSDepStep']].to_xarray()
        indexer_arr = flights[['arr_longitude', 'arr_latitude','UTC_CRSArrTime', 'UTC_CRSArrBaseline' , 'UTC_CRSArrStep']].to_xarray()
        uv_data_dep = weather[1].sel(time = indexer_dep.UTC_CRSDepTime, latitude = indexer_dep.dep_latitude, longitude = indexer_dep.dep_longitude, method = 'nearest')
        uv_data_arr = weather[1].sel(time = indexer_arr.UTC_CRSArrTime, latitude = indexer_arr.arr_latitude, longitude = indexer_arr.arr_longitude, method = 'nearest')
        step_data_dep = weather[0].sel(time = indexer_dep.UTC_CRSDepBaseline, step = indexer_dep.UTC_CRSDepStep , latitude = indexer_dep.dep_latitude, longitude = indexer_dep.dep_longitude, method = 'nearest')
        step_data_arr = weather[0].sel(time = indexer_arr.UTC_CRSArrBaseline, step = indexer_arr.UTC_CRSArrStep , latitude = indexer_arr.arr_latitude, longitude = indexer_arr.arr_longitude, method = 'nearest')
        uv_data_dep_df = uv_data_dep.to_dataframe().rename(columns={'u10': 'dep_u10', 'v10': 'dep_v10'})
        step_data_dep_df = step_data_dep.to_dataframe().rename(columns={'tp': 'dep_tp', 'mxtpr': 'dep_mxtpr', 'fg10' : 'dep_fg10', 'mx2t' : 'dep_mx2t', 'mn2t' : 'dep_mn2t'})
        uv_data_arr_df = uv_data_arr.to_dataframe().rename(columns={'u10': 'arr_u10', 'v10': 'arr_v10'})
        step_data_arr_df = step_data_arr.to_dataframe().rename(columns={'tp': 'arr_tp', 'mxtpr': 'arr_mxtpr', 'fg10' : 'arr_fg10', 'mx2t' : 'arr_mx2t', 'mn2t' : 'arr_mn2t'})
        combined_data = pd.concat([flights, step_data_dep_df[['dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_mx2t', 'dep_mn2t']], uv_data_dep_df[['dep_u10','dep_v10']], 
                           step_data_arr_df[['arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_mx2t', 'arr_mn2t']], uv_data_arr_df[['arr_u10','arr_v10']]], axis =1)
        
        # Converts appropriate column types to less memory-intensive options and saves 
        for column in combined_data.columns:
            if combined_data[column].dtype == 'int64':
                combined_data[column] = combined_data[column].astype('int32')
            if combined_data[column].dtype == 'float64':
                combined_data[column] = combined_data[column].astype('float32')
        combined_data.to_csv(path_dump + '2020_12_with_weather_fixed.csv')
        

    elif year == 2025:
        weather = cfgrib.open_datasets(path_grib + '2025_jan_to_nov_weather.grib')
        weather_step = weather[0]
        weather_step = weather[1]
        for month in range(1,12):
            flights = pd.read_csv(path_flights + f'cleaned_2025_{month}.csv')
            flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
            flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
            in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
            flights = flights[in_window_dep]
            in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
            flights = flights[in_window_arr]
    
            # Creates UTC time datetimes for indexing into GRIB
            flights[['UTC_CRSDepTime', 'UTC_CRSDepBaseline' , 'UTC_CRSDepStep']] = flights.apply(lambda row: convert_date_step(row['Year'], row['Month'], row['DayofMonth'], row['CRSDepTime'], row['Origin']), 
                                                                                         axis=1, result_type = 'expand')
            flights[['UTC_CRSArrTime', 'UTC_CRSArrBaseline' , 'UTC_CRSArrStep']] = flights.apply(lambda row: convert_arr_date_step(row['Year'], row['Month'], row['DayofMonth'], row['CRSDepTime'], row['CRSArrTime'], row['Origin']), 
                                                                                         axis=1, result_type = 'expand')
    
            # Creates xarray versions of our dataset, compiles the weather data with same index into new xrrays, transform then back into pandas and merges
            indexer_dep = flights[['dep_longitude', 'dep_latitude','UTC_CRSDepTime', 'UTC_CRSDepBaseline' , 'UTC_CRSDepStep']].to_xarray()
            indexer_arr = flights[['arr_longitude', 'arr_latitude','UTC_CRSArrTime', 'UTC_CRSArrBaseline' , 'UTC_CRSArrStep']].to_xarray()
            uv_data_dep = weather[1].sel(time = indexer_dep.UTC_CRSDepTime, latitude = indexer_dep.dep_latitude, longitude = indexer_dep.dep_longitude, method = 'nearest')
            uv_data_arr = weather[1].sel(time = indexer_arr.UTC_CRSArrTime, latitude = indexer_arr.arr_latitude, longitude = indexer_arr.arr_longitude, method = 'nearest')
            step_data_dep = weather[0].sel(time = indexer_dep.UTC_CRSDepBaseline, step = indexer_dep.UTC_CRSDepStep , latitude = indexer_dep.dep_latitude, longitude = indexer_dep.dep_longitude, method = 'nearest')
            step_data_arr = weather[0].sel(time = indexer_arr.UTC_CRSArrBaseline, step = indexer_arr.UTC_CRSArrStep , latitude = indexer_arr.arr_latitude, longitude = indexer_arr.arr_longitude, method = 'nearest')
            uv_data_dep_df = uv_data_dep.to_dataframe().rename(columns={'u10': 'dep_u10', 'v10': 'dep_v10'})
            step_data_dep_df = step_data_dep.to_dataframe().rename(columns={'tp': 'dep_tp', 'mxtpr': 'dep_mxtpr', 'fg10' : 'dep_fg10', 'mx2t' : 'dep_mx2t', 'mn2t' : 'dep_mn2t'})
            uv_data_arr_df = uv_data_arr.to_dataframe().rename(columns={'u10': 'arr_u10', 'v10': 'arr_v10'})
            step_data_arr_df = step_data_arr.to_dataframe().rename(columns={'tp': 'arr_tp', 'mxtpr': 'arr_mxtpr', 'fg10' : 'arr_fg10', 'mx2t' : 'arr_mx2t', 'mn2t' : 'arr_mn2t'})
            combined_data = pd.concat([flights, step_data_dep_df[['dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_mx2t', 'dep_mn2t']], uv_data_dep_df[['dep_u10','dep_v10']], 
                               step_data_arr_df[['arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_mx2t', 'arr_mn2t']], uv_data_arr_df[['arr_u10','arr_v10']]], axis =1)
            
            # Converts appropriate column types to less memory-intensive options and saves 
            for column in combined_data.columns:
                if combined_data[column].dtype == 'int64':
                    combined_data[column] = combined_data[column].astype('int32')
                if combined_data[column].dtype == 'float64':
                    combined_data[column] = combined_data[column].astype('float32')
            combined_data.to_csv(path_dump + f'2025_{month}_with_weather_fixed.csv')
                
            

    else: 
        weather = cfgrib.open_datasets(path_grib + f'{year}_jan_to_dec_weather.grib')
        weather_step = weather[0]
        weather_step = weather[1]
        for month in range(1,13):
            flights = pd.read_csv(path_flights + f'cleaned_{year}_{month}.csv')
            flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
            flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
            in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
            flights = flights[in_window_dep]
            in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
            flights = flights[in_window_arr]
    
            # Creates UTC time datetimes for indexing into GRIB
            flights[['UTC_CRSDepTime', 'UTC_CRSDepBaseline' , 'UTC_CRSDepStep']] = flights.apply(lambda row: convert_date_step(row['Year'], row['Month'], row['DayofMonth'], row['CRSDepTime'], row['Origin']), 
                                                                                         axis=1, result_type = 'expand')
            flights[['UTC_CRSArrTime', 'UTC_CRSArrBaseline' , 'UTC_CRSArrStep']] = flights.apply(lambda row: convert_arr_date_step(row['Year'], row['Month'], row['DayofMonth'], row['CRSDepTime'], row['CRSArrTime'], row['Origin']), 
                                                                                         axis=1, result_type = 'expand')
    
            # Creates xarray versions of our dataset, compiles the weather data with same index into new xrrays, transform then back into pandas and merges
            indexer_dep = flights[['dep_longitude', 'dep_latitude','UTC_CRSDepTime', 'UTC_CRSDepBaseline' , 'UTC_CRSDepStep']].to_xarray()
            indexer_arr = flights[['arr_longitude', 'arr_latitude','UTC_CRSArrTime', 'UTC_CRSArrBaseline' , 'UTC_CRSArrStep']].to_xarray()
            uv_data_dep = weather[1].sel(time = indexer_dep.UTC_CRSDepTime, latitude = indexer_dep.dep_latitude, longitude = indexer_dep.dep_longitude, method = 'nearest')
            uv_data_arr = weather[1].sel(time = indexer_arr.UTC_CRSArrTime, latitude = indexer_arr.arr_latitude, longitude = indexer_arr.arr_longitude, method = 'nearest')
            step_data_dep = weather[0].sel(time = indexer_dep.UTC_CRSDepBaseline, step = indexer_dep.UTC_CRSDepStep , latitude = indexer_dep.dep_latitude, longitude = indexer_dep.dep_longitude, method = 'nearest')
            step_data_arr = weather[0].sel(time = indexer_arr.UTC_CRSArrBaseline, step = indexer_arr.UTC_CRSArrStep , latitude = indexer_arr.arr_latitude, longitude = indexer_arr.arr_longitude, method = 'nearest')
            uv_data_dep_df = uv_data_dep.to_dataframe().rename(columns={'u10': 'dep_u10', 'v10': 'dep_v10'})
            step_data_dep_df = step_data_dep.to_dataframe().rename(columns={'tp': 'dep_tp', 'mxtpr': 'dep_mxtpr', 'fg10' : 'dep_fg10', 'mx2t' : 'dep_mx2t', 'mn2t' : 'dep_mn2t'})
            uv_data_arr_df = uv_data_arr.to_dataframe().rename(columns={'u10': 'arr_u10', 'v10': 'arr_v10'})
            step_data_arr_df = step_data_arr.to_dataframe().rename(columns={'tp': 'arr_tp', 'mxtpr': 'arr_mxtpr', 'fg10' : 'arr_fg10', 'mx2t' : 'arr_mx2t', 'mn2t' : 'arr_mn2t'})
            combined_data = pd.concat([flights, step_data_dep_df[['dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_mx2t', 'dep_mn2t']], uv_data_dep_df[['dep_u10','dep_v10']], 
                               step_data_arr_df[['arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_mx2t', 'arr_mn2t']], uv_data_arr_df[['arr_u10','arr_v10']]], axis =1)
            
            # Converts appropriate column types to less memory-intensive options and saves 
            for column in combined_data.columns:
                if combined_data[column].dtype == 'int64':
                    combined_data[column] = combined_data[column].astype('int32')
                if combined_data[column].dtype == 'float64':
                    combined_data[column] = combined_data[column].astype('float32')
            combined_data.to_csv(path_dump + f'{year}_{month}_with_weather_fixed.csv')
        